In [ ]:
from analy_alphas import *
from tushare_util import get_hs300_stocks

start = 2025
end = 2026
alpha_name = 'Alphas191'
for year in range(start, end + 1):  # 2013-2023, including 2022 and 2023.
    list_assets, df_assets = get_hs300_stocks(f'{year}-01-01')
    generate_year_report(alpha_name, list_assets, year)

In [ ]:
from pathlib import Path

from analy_alphas import *

ret = compare_factor('Alphas191', 2022, 2026)
ranked = ret.sort_values(ret.columns[-1], ascending=False)
ranked.columns.name = None

table = ranked.reset_index().to_html(index=False, border=0, classes='result', float_format=lambda x: f'{x:.4f}')
page = """<!DOCTYPE html>
<html lang="zh-CN">
<head>
<meta charset="utf-8">
<title>因子对比</title>
<style>
body { font-family: "Microsoft YaHei", sans-serif; margin: 24px; }
table { border-collapse: collapse; }
th, td { border: 1px solid #ccc; padding: 4px 8px; text-align: right; }
th { cursor: pointer; background: #f3f3f3; position: sticky; top: 0; }
td:first-child, th:first-child { text-align: left; }
tr:nth-child(even) { background: #fafafa; }
</style>
</head>
<body>
<p>点击表头排序。默认按最后一列从高到低。</p>
__TABLE__
<script>
const table = document.querySelector("table");
const headers = [...table.tHead.rows[0].cells];
let sortCol = headers.length - 1;
let sortAsc = false;
function cellValue(row, col) {
  const text = row.cells[col].innerText.trim();
  const num = Number(text);
  return Number.isNaN(num) ? text : num;
}
function sortBy(col, asc) {
  const body = table.tBodies[0];
  const rows = [...body.rows];
  rows.sort((a, b) => {
    const va = cellValue(a, col);
    const vb = cellValue(b, col);
    if (va < vb) return asc ? -1 : 1;
    if (va > vb) return asc ? 1 : -1;
    return 0;
  });
  rows.forEach(row => body.appendChild(row));
}
headers.forEach((th, i) => {
  th.addEventListener("click", () => {
    if (sortCol === i) {
      sortAsc = !sortAsc;
    } else {
      sortCol = i;
      sortAsc = false;
    }
    sortBy(sortCol, sortAsc);
  });
});
sortBy(sortCol, sortAsc);
</script>
</body>
</html>
"""
html_path = Path('analysis/compare_result.html')
html_path.write_text(page.replace('__TABLE__', table), encoding='utf-8')
ranked.to_csv('analysis/compare_result.csv')
print(html_path.resolve())
ranked
